# RF: NFL-Way Text Cleaning

Standard NLP pipelines throw away signal that matters in scouting:

| Problem | Example | Fix |
|---------|---------|-----|
| NLTK drops directional adjectives | "low" in "low pad level" → gone | **Un-stop** high-value directional words |
| Generic filler clogs the vocabulary | "prospect", "player", "shows" in every report | **Custom stop words** for scouting domain |
| Compound phrases split into noise | "change of direction" → 3 useless unigrams | **Phrase stitching**: `change_of_direction` |

This notebook applies all three fixes and inspects the resulting vocabulary before any modeling.

## 1. Imports & Controls

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',  quiet=True)
nltk.download('punkt',    quiet=True)
warnings.filterwarnings('ignore')

# ── Controls ──────────────────────────────────────────────────────────────────
YEAR_MIN   = 2014
YEAR_MAX   = 2021
GRADE_MAX  = 6.4
TARGET     = 'made_it_contract'
RANDOM_STATE = 42

# Outcome-leaking phrases removed before any processing
PHRASE_BLOCKLIST = [
    'undrafted free agent', 'practice squad', 'free agent', 'early starter',
    'pro bowl', 'late round', 'undrafted free', 'make roster', 'rostered',
]

# Bigram discovery: minimum corpus frequency to consider a phrase
AUTO_PHRASE_MIN_FREQ = 15
AUTO_PHRASE_TOP_N    = 60

## 2. Load & Combine Text

In [ ]:
df = pd.read_csv('../data/processed/draft_enriched_with_contracts.csv')
df = df[(df['year'] >= YEAR_MIN) & (df['year'] <= YEAR_MAX)].copy()
df = df[df['grade'] <= GRADE_MAX].copy()
df = df.dropna(subset=[TARGET, 'grade', 'Pos_Group']).copy()
df[TARGET] = df[TARGET].astype(int)

# Combine all three text fields (same convention as other RF notebooks)
def combine_text(row):
    parts = [str(row.get(c, '') or '') for c in ['overview', 'strengths', 'weaknesses']]
    return ' '.join(p for p in parts if p.strip())

df['combined_text'] = df.apply(combine_text, axis=1)
df = df[df['combined_text'].str.strip() != ''].reset_index(drop=True)

print(f'Players: {len(df)}')
print(f'Positive rate ({TARGET}): {df[TARGET].mean():.1%}')
print(f'\nPer Pos_Group:')
print(df['Pos_Group'].value_counts().to_string())

## 3. NFL-Way Cleaning

Three steps applied in order:
1. **Custom stop words** — un-stop directional adjectives, add domain-filler stops
2. **Phrase stitching** — Pass A (curated NFL terms) + Pass B (data-driven discovery)
3. **Preprocessing function** — regex → tokenize → filter → lemmatize

### 3a. Custom Stop Words

In [ ]:
# Words NLTK stops that we WANT to keep — directional / physical descriptors
KEEP_WORDS = {
    'high', 'low', 'heavy', 'light', 'deep', 'short', 'long', 'wide',
    'hard', 'soft', 'strong', 'quick', 'good', 'great', 'up', 'down',
    'off', 'out', 'over', 'through', 'above', 'below',
}

# Words to ADD to stop list — appear in nearly every scouting report, carry no discriminative signal
CUSTOM_STOPS = {
    'prospect', 'player', 'players', 'show', 'shows', 'need', 'needs',
    'ability', 'also', 'often', 'must', 'well', 'still', 'use', 'get',
    'make', 'look', 'help', 'work', 'time', 'year', 'team', 'game',
    'continue', 'develop', 'development', 'nfl', 'draft', 'college',
    'level', 'type', 'project', 'potential', 'upside', 'ceiling',
}

_base_stops = set(stopwords.words('english'))
NFL_STOPWORDS = (_base_stops - KEEP_WORDS) | CUSTOM_STOPS

print(f'Base NLTK stop words:  {len(_base_stops)}')
print(f'Words un-stopped:      {len(KEEP_WORDS & _base_stops)}')
print(f'Custom words added:    {len(CUSTOM_STOPS)}')
print(f'Final NFL stop list:   {len(NFL_STOPWORDS)}')
print(f'\nVerify kept words still in vocab (not stopped):')
print(sorted(KEEP_WORDS & _base_stops))

### 3b. Phrase Stitching — Pass A: Curated NFL Compound Terms

These are applied as simple string replacements **before** tokenization,
so the phrase context is preserved. Longest phrases run first to prevent partial matches.

In [ ]:
# key = phrase (lowercase), value = replacement token
# Sorted longest-first at apply time to prevent partial matches
_NFL_PHRASE_MAP_RAW = {
    # ── Tri-grams (apply before bi-grams) ─────────────────────────────────────
    'change of direction':  'change_of_direction',
    'low pad level':        'low_pad_level',
    'run after catch':      'run_after_catch',
    'yards after contact':  'yards_after_contact',
    'yards after catch':    'yards_after_catch',
    'off the line':         'off_the_line',
    'off the ball':         'off_the_ball',
    'point of attack':      'point_of_attack',
    'get off the':          'get_off',        # catches 'get off the line' etc.
    # ── Bi-grams ──────────────────────────────────────────────────────────────
    'pad level':            'pad_level',
    'pass rush':            'pass_rush',
    'pass rusher':          'pass_rusher',
    'rush ability':         'rush_ability',
    'open field':           'open_field',
    'high cut':             'high_cut',
    'press coverage':       'press_coverage',
    'man coverage':         'man_coverage',
    'zone coverage':        'zone_coverage',
    'hand fighting':        'hand_fighting',
    'block shedding':       'block_shedding',
    'first step':           'first_step',
    'second level':         'second_level',
    'heavy hands':          'heavy_hands',
    'soft hands':           'soft_hands',
    'strong hands':         'strong_hands',
    'anchor strength':      'anchor_strength',
    'three down':           'three_down',
    'red zone':             'red_zone',
    'snap count':           'snap_count',
    'two gap':              'two_gap',
    'one gap':              'one_gap',
    'get off':              'get_off',
    'pass protection':      'pass_protection',
    'run blocking':         'run_blocking',
    'route running':        'route_running',
    'ball carrier':         'ball_carrier',
    'ball hawk':            'ball_hawk',
    'pass coverage':        'pass_coverage',
    'hip flexibility':      'hip_flexibility',
    'closing speed':        'closing_speed',
    'contact balance':      'contact_balance',
    'body control':         'body_control',
    'burst speed':          'burst_speed',
    'lateral quickness':    'lateral_quickness',
    'short area':           'short_area',
    'high motor':           'high_motor',
    'low center':           'low_center',
    'hand strength':        'hand_strength',
    'quick twitch':         'quick_twitch',
    'top end':              'top_end',
}

# Sort by length descending so tri-grams are replaced before bi-grams
NFL_PHRASE_MAP = dict(
    sorted(_NFL_PHRASE_MAP_RAW.items(), key=lambda x: len(x[0]), reverse=True)
)

print(f'Curated NFL phrases: {len(NFL_PHRASE_MAP)}')
print('\nSample (longest first):')
for k, v in list(NFL_PHRASE_MAP.items())[:8]:
    print(f'  "{k}" → "{v}"')

### 3c. Preprocessing Function

In [ ]:
lemmatizer = WordNetLemmatizer()

def nfl_preprocess(text: str, phrase_map: dict = NFL_PHRASE_MAP) -> str:
    """NFL-aware preprocessing pipeline."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()

    # 1. Remove outcome-leaking phrases (longest first)
    for phrase in sorted(PHRASE_BLOCKLIST, key=len, reverse=True):
        text = text.replace(phrase, ' ')

    # 2. Normalize hyphens/dashes → spaces BEFORE phrase stitching
    #    so "pass-rush", "pass–rush", "pass—rush" all become "pass rush"
    #    and get caught by the phrase map below
    text = re.sub(r'[-\u2013\u2014]', ' ', text)

    # 3. Stitch curated NFL compound phrases
    for phrase, replacement in phrase_map.items():
        text = text.replace(phrase, replacement)

    # 4. Regex: keep letters, underscores (stitched tokens), spaces
    text = re.sub(r'[^a-z_\s]', ' ', text)

    # 5. Tokenize
    tokens = text.split()

    # 6. Stop word filter — always keep stitched tokens (contain '_')
    tokens = [t for t in tokens if '_' in t or t not in NFL_STOPWORDS]

    # 7. Lemmatize non-stitched tokens only
    tokens = [t if '_' in t else lemmatizer.lemmatize(t) for t in tokens]

    # 8. Drop short tokens (keep stitched phrases regardless)
    tokens = [t for t in tokens if len(t) > 1]

    return ' '.join(tokens)


# Also define the standard NLTK pipeline for side-by-side comparison
_std_stops = set(stopwords.words('english'))

def standard_preprocess(text: str) -> str:
    """Standard NLTK pipeline (same as rf_text_made_it) for comparison."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    for phrase in sorted(PHRASE_BLOCKLIST, key=len, reverse=True):
        text = text.replace(phrase, ' ')
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [
        lemmatizer.lemmatize(w)
        for w in text.split()
        if w not in _std_stops and len(w) > 1
    ]
    return ' '.join(tokens)


print('Applying NFL preprocessing...')
df['text_nfl']      = df['combined_text'].apply(nfl_preprocess)
df['text_standard'] = df['combined_text'].apply(standard_preprocess)
df = df[df['text_nfl'] != ''].reset_index(drop=True)
print(f'Done. {len(df)} players with non-empty NFL-cleaned text.')

# Quick spot-check: verify hyphenated phrases are stitched correctly
test_cases = [
    'pass-rush talent with great get-off',
    'high-motor player with low pad-level issues',
    'route-running and man-coverage ability',
    'heavy-handed blocker with body-control problems',
]
print('\nHyphen normalization spot-check:')
for t in test_cases:
    print(f'  IN:  {t}')
    print(f'  OUT: {nfl_preprocess(t)}')
    print()

### 3d. Phrase Stitching — Pass B: Data-Driven Bigram Discovery

Run `BigramCollocationFinder` on the corpus **after** Pass A stitching to discover
high-PMI bigrams the curated list missed. **Review the output** — add keepers to
`NFL_PHRASE_MAP` above and re-run if desired.

In [ ]:
# Tokenise the NFL-cleaned text for collocation discovery
# Only use non-stitched tokens for bigram finding (stitched tokens already handled)
_all_token_lists = [
    [t for t in text.split() if '_' not in t]   # exclude already-stitched phrases
    for text in df['text_nfl']
]

finder = BigramCollocationFinder.from_documents(_all_token_lists)
finder.apply_freq_filter(AUTO_PHRASE_MIN_FREQ)

# Score by PMI
scored = finder.score_ngrams(BigramAssocMeasures.pmi)

# Filter: both tokens must be non-stopword, alphabetic, len > 2
already_stitched = set(NFL_PHRASE_MAP.keys())
_curated_tokens  = {t for phrase in already_stitched for t in phrase.split()}

auto_phrases = [
    (w1, w2, score)
    for (w1, w2), score in scored
    if  w1 not in NFL_STOPWORDS
    and w2 not in NFL_STOPWORDS
    and w1.isalpha() and w2.isalpha()
    and len(w1) > 2  and len(w2) > 2
    and f'{w1} {w2}' not in already_stitched
][:AUTO_PHRASE_TOP_N]

auto_phrase_df = pd.DataFrame(auto_phrases, columns=['word1', 'word2', 'pmi'])
auto_phrase_df['phrase']      = auto_phrase_df['word1'] + ' ' + auto_phrase_df['word2']
auto_phrase_df['token']       = auto_phrase_df['word1'] + '_' + auto_phrase_df['word2']
auto_phrase_df['corpus_freq'] = auto_phrase_df.apply(
    lambda r: finder.ngram_fd[(r['word1'], r['word2'])], axis=1
)

print(f'Auto-discovered bigrams (freq ≥ {AUTO_PHRASE_MIN_FREQ}, top {AUTO_PHRASE_TOP_N} by PMI):')
print('Review these — add useful ones to NFL_PHRASE_MAP above and re-run\n')
print(auto_phrase_df[['phrase', 'token', 'pmi', 'corpus_freq']]
      .sort_values('pmi', ascending=False)
      .to_string(index=False))

## 4. Vocabulary Inspection

What does the cleaned corpus actually look like? No model — just the tokens.

In [ ]:
# ── Token frequency across all documents ──────────────────────────────────────
all_tokens = [t for text in df['text_nfl'] for t in text.split()]
token_counts = Counter(all_tokens)

# Split stitched vs regular
stitched_counts = {k: v for k, v in token_counts.items() if '_' in k}
regular_counts  = {k: v for k, v in token_counts.items() if '_' not in k}

print(f'Total tokens: {len(all_tokens):,}')
print(f'Unique tokens: {len(token_counts):,}  '
      f'(regular: {len(regular_counts):,}, stitched phrases: {len(stitched_counts):,})')

print('\n── Top 50 regular tokens (NFL-cleaned) ──────────────────────────────────')
top_regular = pd.Series(regular_counts).sort_values(ascending=False).head(50)
print(top_regular.to_frame('count').to_string())

print('\n── All stitched phrase tokens (Pass A curated + survivors) ───────────────')
stitched_series = pd.Series(stitched_counts).sort_values(ascending=False)
print(stitched_series.to_frame('count').to_string())

In [ ]:
# ── Sanity checks ─────────────────────────────────────────────────────────────
should_be_absent = ['prospect', 'player', 'players', 'show', 'shows', 'need', 'needs',
                    'ability', 'nfl', 'draft', 'college']
should_be_present = ['high', 'low', 'heavy', 'deep', 'hard', 'soft', 'strong', 'quick',
                     'light', 'long', 'wide']

print('── Words that should be ABSENT (custom stops) ────────────────────────────')
for w in should_be_absent:
    cnt = token_counts.get(w, 0)
    status = '✓ ABSENT' if cnt == 0 else f'✗ PRESENT ({cnt}x)'
    print(f'  {w:20s} {status}')

print('\n── Words that should be PRESENT (un-stopped directionals) ───────────────')
for w in should_be_present:
    cnt = token_counts.get(w, 0)
    status = f'✓ PRESENT ({cnt}x)' if cnt > 0 else '✗ ABSENT'
    print(f'  {w:20s} {status}')

In [ ]:
# ── Side-by-side sample: raw → standard NLTK → NFL-clean ──────────────────────
# Pick players whose raw text contains at least one curated phrase
phrase_keywords = ['pad level', 'change of direction', 'heavy hands',
                   'pass rush', 'run after catch', 'press coverage', 'high motor']

sample_indices = []
for kw in phrase_keywords:
    matches = df[df['combined_text'].str.lower().str.contains(kw, na=False)].index
    if len(matches) > 0:
        sample_indices.append(matches[0])
    if len(sample_indices) >= 5:
        break

# Also grab a couple of random players
import random; random.seed(42)
sample_indices += random.sample(list(df.index), min(3, len(df)))
sample_indices = list(dict.fromkeys(sample_indices))[:6]   # deduplicate, cap at 6

print('Side-by-side comparison — raw vs standard NLTK vs NFL-clean')
print('=' * 100)
for idx in sample_indices:
    row = df.loc[idx]
    print(f"\n[{row['Pos_Group']}] {row.get('player_name', idx)}  "
          f"grade={row.get('grade', '?')}  {TARGET}={row[TARGET]}")
    print(f"  RAW:      {str(row['combined_text'])[:200]}")
    print(f"  STANDARD: {row['text_standard'][:200]}")
    print(f"  NFL-CLEAN:{row['text_nfl'][:200]}")
    print('-' * 100)